# Phase 2: Cleaning and Feature Engineering

## Project context

This notebook parses collected Philippines macro files, inventories additional local Excel sources, standardizes dates and columns, and creates an inflation-first monthly indicator table. It does not train forecasts or build the dashboard.

## Phase 2 objective

- Inspect raw Excel files under `data/raw/`.
- Parse BSP monthly inflation and peso-dollar data where formats are clear.
- Parse annual World Bank context indicators.
- Build a monthly inflation-first table with simple lag, rolling, and change features.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR
from src.cleaning import (
    list_excel_files,
    inspect_excel_workbook,
    parse_bsp_inflation,
    parse_bsp_peso_dollar,
    parse_world_bank_csv,
    save_clean_dataset,
)
from src.features import build_inflation_feature_table
from src.visualization import (
    plot_time_series,
    plot_indicator_correlation,
    plot_missingness_summary,
)

INDICATORS_DIR = OUTPUTS_DIR / "indicators"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
INDICATORS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def save_plotly_or_pillow(fig, output_path, chart_type, data):
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception:
        draw_basic_png(output_path, chart_type, data)
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly static export was unavailable.", fill="#555555", font=font)
    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "line":
        frame = data.dropna().copy()
        if len(frame) > 260:
            frame = frame.iloc[np.linspace(0, len(frame) - 1, 260).astype(int)]
        values = frame.iloc[:, 1].astype(float).to_numpy()
        if len(values) > 1:
            vmin, vmax = np.nanmin(values), np.nanmax(values)
            if np.isclose(vmin, vmax):
                vmin, vmax = vmin - 1, vmax + 1
            points = []
            for i, value in enumerate(values):
                x = left + (right - left) * i / max(len(values) - 1, 1)
                y = bottom - (bottom - top) * ((value - vmin) / (vmax - vmin))
                points.append((x, y))
            draw.line(points, fill="#1f77b4", width=2)
    elif chart_type == "bar":
        frame = data.copy()
        values = frame["missing_values"].astype(float).to_numpy()
        labels = frame["column"].astype(str).to_list()
        vmax = max(np.nanmax(values), 1)
        bar_width = (right - left) / max(len(values), 1) * 0.7
        for i, value in enumerate(values):
            x = left + (right - left) * (i + 0.5) / len(values)
            y = bottom - (bottom - top) * (value / vmax)
            draw.rectangle((x - bar_width / 2, y, x + bar_width / 2, bottom), fill="#1f77b4")
            draw.text((x - 30, bottom + 8), labels[i][:10], fill="black", font=font)
    elif chart_type == "heatmap":
        corr = data.copy()
        labels = corr.columns.to_list()
        n = len(labels)
        cell = min((right - left) / max(n, 1), (bottom - top) / max(n, 1))
        for i, row in enumerate(labels):
            for j, col in enumerate(labels):
                value = corr.loc[row, col]
                red = int(255 * max(value, 0))
                blue = int(255 * abs(min(value, 0)))
                green = int(220 * (1 - abs(value)))
                x0 = left + j * cell
                y0 = top + i * cell
                draw.rectangle((x0, y0, x0 + cell, y0 + cell), fill=(red, green, blue), outline="white")
                draw.text((x0 + 4, y0 + 4), f"{value:.2f}", fill="black", font=font)
    image.save(output_path)


def quality_summary(dataset, df, date_col="date", status="created", notes=""):
    start_date = None
    end_date = None
    duplicate_dates = 0
    if date_col in df.columns:
        if date_col == "year":
            years = pd.to_numeric(df[date_col], errors="coerce")
            start_date = int(years.min()) if years.notna().any() else None
            end_date = int(years.max()) if years.notna().any() else None
            duplicate_dates = int(years.duplicated().sum())
        else:
            dates = pd.to_datetime(df[date_col], errors="coerce")
            start_date = dates.min()
            end_date = dates.max()
            duplicate_dates = int(dates.duplicated().sum())
    return {
        "dataset": dataset,
        "rows": len(df),
        "columns": len(df.columns),
        "start_date": start_date,
        "end_date": end_date,
        "missing_values_total": int(df.isna().sum().sum()),
        "duplicate_dates": duplicate_dates,
        "status": status,
        "notes": notes,
    }


## Raw file inventory

In [3]:
excel_files = list_excel_files(RAW_DATA_DIR)
excel_inventory = pd.DataFrame([inspect_excel_workbook(path) for path in excel_files])
excel_inventory_path = INDICATORS_DIR / "raw_excel_inventory.csv"
excel_inventory.to_csv(excel_inventory_path, index=False)
display(excel_inventory)
print(f"Excel files inspected: {len(excel_inventory)}")

,file_path,file_name,file_type,sheet_count,sheet_names,status,notes
0,/Users/rovs/Documents/New project 2/projects/p...,API_PHL_DS2_en_excel_v2_6947.xls,.xls,3,Data; Metadata - Countries; Metadata - Indicators,inspectable,Workbook opened successfully.
1,/Users/rovs/Documents/New project 2/projects/p...,RERB.xlsx,.xlsx,1,RERB,inspectable,Workbook opened successfully.
2,/Users/rovs/Documents/New project 2/projects/p...,Statistical Tables on March 2026 CPI for All I...,.xlsx,16,table 1; table 2; table 3; table 4; table 5; t...,inspectable,Workbook opened successfully.
3,/Users/rovs/Documents/New project 2/projects/p...,inf_bottom30_2018.xls,.xls,4,Monthly Inflation Rate; Inflation Rate by Comm...,inspectable,Workbook opened successfully.
4,/Users/rovs/Documents/New project 2/projects/p...,infrate2018.xls,.xls,2,Annual; Monthly,inspectable,Workbook opened successfully.
5,/Users/rovs/Documents/New project 2/projects/p...,infrate_comm2018.xls,.xls,2,Annual; Monthly,inspectable,Workbook opened successfully.
6,/Users/rovs/Documents/New project 2/projects/p...,pesodollar.xlsx,.xlsx,3,monthly; annual; daily,inspectable,Workbook opened successfully.
7,/Users/rovs/Documents/New project 2/projects/p...,prices2018.xls,.xls,6,Annual PHL; Annual NCR; Annual AONCR; Monthly ...,inspectable,Workbook opened successfully.
8,/Users/rovs/Documents/New project 2/projects/p...,bsp_inflation_infrate.xls,.xls,2,Annual; Monthly,inspectable,Workbook opened successfully.
9,/Users/rovs/Documents/New project 2/projects/p...,bsp_peso_dollar.xlsx,.xlsx,3,monthly; annual; daily,inspectable,Workbook opened successfully.


Excel files inspected: 10


## Additional local Excel file inspection

Additional local Excel files are inventoried. They are not merged into the core monthly table unless their structure is clearly useful and safely parseable for the inflation-first MVP.

In [4]:
additional_excel_files = [path for path in excel_files if "additional_sources" in str(path)]
for path in additional_excel_files:
    print(path.name)


API_PHL_DS2_en_excel_v2_6947.xls
RERB.xlsx
Statistical Tables on March 2026 CPI for All Income Households (2018=100)_k4r8j.xlsx
inf_bottom30_2018.xls
infrate2018.xls
infrate_comm2018.xls
pesodollar.xlsx
prices2018.xls


## Parse BSP monthly inflation

In [5]:
primary_inflation_path = RAW_DATA_DIR / "bsp_inflation_infrate.xls"
modern_inflation_path = RAW_DATA_DIR / "additional_sources" / "infrate2018.xls"

primary_inflation = parse_bsp_inflation(primary_inflation_path)
inflation_source_used = primary_inflation_path
monthly_inflation = primary_inflation

if modern_inflation_path.exists():
    modern_inflation = parse_bsp_inflation(modern_inflation_path)
    if modern_inflation["date"].max() > primary_inflation["date"].max():
        monthly_inflation = modern_inflation
        inflation_source_used = modern_inflation_path

monthly_inflation_path = PROCESSED_DATA_DIR / "monthly_inflation.csv"
save_clean_dataset(monthly_inflation, monthly_inflation_path)
display(monthly_inflation.head())
display(monthly_inflation.tail())
print(f"Inflation source used: {inflation_source_used}")
print(f"Rows: {len(monthly_inflation):,}; range: {monthly_inflation['date'].min().date()} to {monthly_inflation['date'].max().date()}")

,date,year,month,inflation_rate
0,1958-01-01,1958,1,6.3
1,1958-02-01,1958,2,6.3
2,1958-03-01,1958,3,5.5
3,1958-04-01,1958,4,5.5
4,1958-05-01,1958,5,6.3


,date,year,month,inflation_rate
814,2025-11-01,2025,11,1.5
815,2025-12-01,2025,12,1.8
816,2026-01-01,2026,1,2.0
817,2026-02-01,2026,2,2.4
818,2026-03-01,2026,3,4.1


Inflation source used: /Users/rovs/Documents/New project 2/projects/philippines-macro-nowcasting-dashboard/data/raw/additional_sources/infrate2018.xls
Rows: 819; range: 1958-01-01 to 2026-03-01


## Parse BSP peso-dollar exchange rate

In [6]:
monthly_usd_php = parse_bsp_peso_dollar(RAW_DATA_DIR / "bsp_peso_dollar.xlsx")
monthly_usd_php_path = PROCESSED_DATA_DIR / "monthly_usd_php.csv"
save_clean_dataset(monthly_usd_php, monthly_usd_php_path)
display(monthly_usd_php.head())
display(monthly_usd_php.tail())
print(f"Rows: {len(monthly_usd_php):,}; range: {monthly_usd_php['date'].min().date()} to {monthly_usd_php['date'].max().date()}")

,date,year,month,usd_php
0,1945-01-01,1945,1,2.0
1,1945-02-01,1945,2,2.0
2,1945-03-01,1945,3,2.0
3,1945-04-01,1945,4,2.0
4,1945-05-01,1945,5,2.0


,date,year,month,usd_php
970,2025-11-01,2025,11,58.913550
971,2025-12-01,2025,12,58.848833
972,2026-01-01,2026,1,59.162190
973,2026-02-01,2026,2,58.280263
974,2026-03-01,2026,3,59.406905


Rows: 975; range: 1945-01-01 to 2026-03-01


## Parse World Bank annual context indicators

In [7]:
annual_context_parts = [
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_gdp_growth.csv", "gdp_growth"),
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_unemployment.csv", "unemployment_rate"),
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_inflation_backup.csv", "inflation_backup"),
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_remittances_pct_gdp.csv", "remittances_pct_gdp"),
]

annual_macro_context = annual_context_parts[0]
for part in annual_context_parts[1:]:
    annual_macro_context = annual_macro_context.merge(part, on="year", how="outer")
annual_macro_context = annual_macro_context.sort_values("year").reset_index(drop=True)
annual_macro_context_path = PROCESSED_DATA_DIR / "annual_macro_context.csv"
save_clean_dataset(annual_macro_context, annual_macro_context_path)
display(annual_macro_context.tail())

,year,gdp_growth,unemployment_rate,inflation_backup,remittances_pct_gdp
21,2021,5.714733,3.398,3.927180,9.308929
22,2022,7.580982,2.598,5.821158,9.409771
23,2023,5.518950,2.200,5.978025,8.945518
24,2024,5.692016,2.202,3.212605,8.725710
25,2025,NaN,2.235,NaN,NaN


## Standardize monthly dates

In [8]:
monthly_inflation["date"] = pd.to_datetime(monthly_inflation["date"])
monthly_usd_php["date"] = pd.to_datetime(monthly_usd_php["date"])
monthly_inflation["date"] = monthly_inflation["date"].dt.to_period("M").dt.to_timestamp()
monthly_usd_php["date"] = monthly_usd_php["date"].dt.to_period("M").dt.to_timestamp()
print(monthly_inflation.dtypes)
print(monthly_usd_php.dtypes)

date              datetime64[ns]
year                       int64
month                      int64
inflation_rate           float64
dtype: object
date       datetime64[ns]
year                int64
month               int64
usd_php           float64
dtype: object


## Build inflation-first monthly indicator table

In [9]:
monthly_macro = monthly_inflation.merge(
    monthly_usd_php[["date", "usd_php"]], on="date", how="left"
)
monthly_macro = monthly_macro.sort_values("date").reset_index(drop=True)
display(monthly_macro.head())
display(monthly_macro.tail())

,date,year,month,inflation_rate,usd_php
0,1958-01-01,1958,1,6.3,2.0
1,1958-02-01,1958,2,6.3,2.0
2,1958-03-01,1958,3,5.5,2.0
3,1958-04-01,1958,4,5.5,2.0
4,1958-05-01,1958,5,6.3,2.0


,date,year,month,inflation_rate,usd_php
814,2025-11-01,2025,11,1.5,58.913550
815,2025-12-01,2025,12,1.8,58.848833
816,2026-01-01,2026,1,2.0,59.162190
817,2026-02-01,2026,2,2.4,58.280263
818,2026-03-01,2026,3,4.1,59.406905


## Add lag, rolling, and change features

In [10]:
monthly_macro_features = build_inflation_feature_table(monthly_macro)
monthly_macro_path = PROCESSED_DATA_DIR / "monthly_macro_indicators.csv"
save_clean_dataset(monthly_macro_features, monthly_macro_path)
display(monthly_macro_features.head(10))
display(monthly_macro_features.tail())
monthly_macro_features.columns.tolist()

,date,year,month,inflation_rate,usd_php,inflation_rate_lag_1,inflation_rate_lag_3,inflation_rate_lag_6,inflation_rate_rolling_3,inflation_rate_rolling_6,inflation_rate_change_1,usd_php_lag_1,usd_php_change_1
0,1958-01-01,1958,1,6.3,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1958-02-01,1958,2,6.3,2.0,6.3,NaN,NaN,NaN,NaN,0.0,2.0,0.0
2,1958-03-01,1958,3,5.5,2.0,6.3,NaN,NaN,6.033333,NaN,-0.8,2.0,0.0
3,1958-04-01,1958,4,5.5,2.0,5.5,6.3,NaN,5.766667,NaN,0.0,2.0,0.0
4,1958-05-01,1958,5,6.3,2.0,5.5,6.3,NaN,5.766667,NaN,0.8,2.0,0.0
5,1958-06-01,1958,6,5.4,2.0,6.3,5.5,NaN,5.733333,5.883333,-0.9,2.0,0.0
6,1958-07-01,1958,7,3.8,2.0,5.4,5.5,6.3,5.166667,5.466667,-1.6,2.0,0.0
7,1958-08-01,1958,8,3.0,2.0,3.8,6.3,6.3,4.066667,4.916667,-0.8,2.0,0.0
8,1958-09-01,1958,9,2.3,2.0,3.0,5.4,5.5,3.033333,4.383333,-0.7,2.0,0.0
9,1958-10-01,1958,10,0.7,2.0,2.3,3.8,5.5,2.000000,3.583333,-1.6,2.0,0.0


,date,year,month,inflation_rate,usd_php,inflation_rate_lag_1,inflation_rate_lag_3,inflation_rate_lag_6,inflation_rate_rolling_3,inflation_rate_rolling_6,inflation_rate_change_1,usd_php_lag_1,usd_php_change_1
814,2025-11-01,2025,11,1.5,58.913550,1.7,1.5,1.3,1.633333,1.450000,-0.2,58.298409,0.615141
815,2025-12-01,2025,12,1.8,58.848833,1.5,1.7,1.4,1.666667,1.516667,0.3,58.913550,-0.064717
816,2026-01-01,2026,1,2.0,59.162190,1.8,1.7,0.9,1.766667,1.700000,0.2,58.848833,0.313357
817,2026-02-01,2026,2,2.4,58.280263,2.0,1.5,1.5,2.066667,1.850000,0.4,59.162190,-0.881927
818,2026-03-01,2026,3,4.1,59.406905,2.4,1.8,1.7,2.833333,2.250000,1.7,58.280263,1.126642


['date',
 'year',
 'month',
 'inflation_rate',
 'usd_php',
 'inflation_rate_lag_1',
 'inflation_rate_lag_3',
 'inflation_rate_lag_6',
 'inflation_rate_rolling_3',
 'inflation_rate_rolling_6',
 'inflation_rate_change_1',
 'usd_php_lag_1',
 'usd_php_change_1']

## Data quality review

In [11]:
quality_rows = [
    quality_summary("monthly_inflation", monthly_inflation, notes=f"Source used: {inflation_source_used.name}"),
    quality_summary("monthly_usd_php", monthly_usd_php),
    quality_summary("annual_macro_context", annual_macro_context, date_col="year", notes="Annual World Bank context indicators."),
    quality_summary("monthly_macro_indicators", monthly_macro_features, notes="Inflation-first monthly feature table."),
]
data_quality_summary = pd.DataFrame(quality_rows)
data_quality_summary_path = INDICATORS_DIR / "data_quality_summary.csv"
data_quality_summary.to_csv(data_quality_summary_path, index=False)
display(data_quality_summary)

,dataset,rows,columns,start_date,end_date,missing_values_total,duplicate_dates,status,notes
0,monthly_inflation,819,4,1958-01-01 00:00:00,2026-03-01 00:00:00,0,0,created,Source used: infrate2018.xls
1,monthly_usd_php,975,4,1945-01-01 00:00:00,2026-03-01 00:00:00,0,0,created,
2,annual_macro_context,26,5,2000,2025,3,0,created,Annual World Bank context indicators.
3,monthly_macro_indicators,819,13,1958-01-01 00:00:00,2026-03-01 00:00:00,20,0,created,Inflation-first monthly feature table.


## Initial indicator charts

In [12]:
figure_exports = {}

fig = plot_time_series(monthly_inflation, "date", "inflation_rate", "Philippines Monthly Inflation Rate")
figure_exports["inflation_time_series.png"] = save_plotly_or_pillow(
    fig, FIGURES_DIR / "inflation_time_series.png", "line", monthly_inflation[["date", "inflation_rate"]]
)

fig = plot_time_series(monthly_usd_php, "date", "usd_php", "Monthly USD/PHP Average Exchange Rate")
figure_exports["usd_php_time_series.png"] = save_plotly_or_pillow(
    fig, FIGURES_DIR / "usd_php_time_series.png", "line", monthly_usd_php[["date", "usd_php"]]
)

missing_data = monthly_macro_features.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_values"})
fig = plot_missingness_summary(monthly_macro_features)
figure_exports["inflation_features_missingness.png"] = save_plotly_or_pillow(
    fig, FIGURES_DIR / "inflation_features_missingness.png", "bar", missing_data
)

numeric_cols = monthly_macro_features.select_dtypes(include="number").columns.tolist()
if len(numeric_cols) >= 2:
    corr = monthly_macro_features[numeric_cols].corr()
    fig = plot_indicator_correlation(monthly_macro_features[numeric_cols])
    figure_exports["macro_indicator_correlation.png"] = save_plotly_or_pillow(
        fig, FIGURES_DIR / "macro_indicator_correlation.png", "heatmap", corr
    )

figure_exports

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  v = v.dt.to_pydatetime()
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



{'inflation_time_series.png': 'pillow_fallback',
 'usd_php_time_series.png': 'pillow_fallback',
 'inflation_features_missingness.png': 'pillow_fallback',
 'macro_indicator_correlation.png': 'pillow_fallback'}

## Phase 2 limitations

- Forecasting is not performed in this phase.
- Policy rate data remains manual or future-parser work.
- Additional local Excel files were inventoried, but only the clearly parseable BSP inflation and peso-dollar structures were used in the core table.
- World Bank indicators are annual and are kept as context rather than merged into the monthly feature table.

## Next steps for Phase 3 baseline forecasting

- Define the forecasting target, likely one-month-ahead inflation.
- Split data into train/test periods.
- Build simple baseline models using lag and rolling features.
- Compare results against naive inflation benchmarks.